# Phase 6: compare inference using the saved trained model

The hosted T4 run completed on 24 October 2026: all three batch sizes and 171 tests passed.
See the [reviewed results](https://github.com/WeaversMask/temporal-relational-gnn/blob/main/reports/TRAINED_INFERENCE_STATUS.md).

This notebook measures **reference, existing HGAM trimming and bipartite blocks** using the
unchanged checkpoint from notebook 05. It does not train a model. Each method receives the same
sampled per-query occurrences. The primary comparison is batch size **32**; batch sizes **128 and
512** examine latency, throughput and memory as more queries are processed together. The fixed
8,192-query validation cohort, constant features and relative time match the parent training run.
The test split stays untouched.

The profiler first verifies the saved checkpoint, its input contract and the parent validation
results. It checks prediction agreement outside timing, then records **20 measured repeats after
5 warmups per batch and execution method**, plus raw measurements and memory. The online boundary
starts with a resident graph and includes sampling, feature preparation, transfer and prediction;
downloads, graph construction, validation replay and correctness checks stay outside that boundary.
Results describe this checkpoint, cohort and hosted-snapshot assumptions.

1. Extract **`checkpoints/training.pt`** from the notebook 05 ZIP
   `trgnn-training-run-20261024T073010Z-1f9800.zip`. Keep that earlier ZIP.
2. Save a copy of this notebook to Drive. This notebook uses temporary VM storage and does not
   mount Drive. Disconnect other managed runtimes, select **T4 GPU**, and connect.
3. Enable your existing read-only `GITHUB_TOKEN` in Secrets. Enter current budget readings when
   available; leave an unavailable observation `None`.
4. Run cells in order. In the checkpoint cell, upload that single `training.pt` when prompted.
   Its SHA256 must match the reviewed parent report before data download or profiling begins.
5. If a step fails, run the ending-balance and ZIP-export cells. Export before disconnecting.

The ZIP includes the input checkpoint, configuration, logs, report and individual measured-batch
artifacts. It excludes the database. Keep the ZIP so completed measurements remain reviewable.
After a stop on the same VM, set `RESUME = True` and rerun cells; completed batches are reused only
when their provenance and checksums verify. The parent validation replay still runs on resume.

In [ ]:
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
# Set True after checking that other managed runtimes are disconnected.
ISOLATED_RUNTIME_CONFIRMED = False
RESUME = False
CHECKPOINT_INPUT = None  # Own notebook 05 training.pt path, or None to upload it below.
RESUME_ARTIFACTS_DIR = None  # For a fresh VM: extracted prior profiling ZIP/batches path.
RESUME_RUN_ID = None  # On a fresh VM, optionally retain the ID from your earlier exported ZIP.
RESERVE_UNITS = 50
PROJECT_ROOT = Path("/content/temporal-relational-gnn")
DATA_DIR = PROJECT_ROOT / "data/rel-stack"
PROJECT_REF = "main"  # Same-folder reruns retain the first successful profiling checkout.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

# Preserve the same run folder if this cell is rerun in this notebook session.
if "_INFERENCE_RUN_DIR" not in globals():
    if RESUME_RUN_ID is not None:
        if not RESUME or not isinstance(RESUME_RUN_ID, str) or not RESUME_RUN_ID:
            raise ValueError("RESUME_RUN_ID needs RESUME=True and the earlier run ID")
        if any(character not in "0123456789TZ-abcdef" for character in RESUME_RUN_ID):
            raise ValueError("Use only the run ID from the earlier exported ZIP filename")
    _INFERENCE_RUN_ID = RESUME_RUN_ID or (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    )
    _INFERENCE_RUN_DIR = Path("/content/temporary-trgnn-trained-inference/runs") / _INFERENCE_RUN_ID
    _INFERENCE_RUN_DIR.mkdir(parents=True, exist_ok=False)
run_id = _INFERENCE_RUN_ID
RUN_DIR = _INFERENCE_RUN_DIR
CHECKPOINT_PATH = RUN_DIR / "checkpoints/training.pt"
TRAINING_REPORT = PROJECT_ROOT / "reports/colab/20261024T073010Z-1f9800/training-run.json"
print(f"Run directory: {RUN_DIR}", flush=True)
print(f"Checkpoint: {CHECKPOINT_PATH}", flush=True)


def run_logged(arguments, log_name, *, cwd=None, env=None):
    with (RUN_DIR / log_name).open("a") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT if cwd is None else cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)


def observed_nonnegative_number(name, value):
    if value is not None and (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or value < 0
    ):
        raise ValueError(f"{name} must be an observed non-negative number or None")


observed_nonnegative_number("COMPUTE_UNITS_PER_HOUR", COMPUTE_UNITS_PER_HOUR)
observed_nonnegative_number("COMPUTE_UNITS_REMAINING", COMPUTE_UNITS_REMAINING)
budget_path = RUN_DIR / "colab-budget.json"
if not budget_path.exists():
    budget_path.write_text(
        json.dumps(
            {
                "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
                "units_remaining_observed": COMPUTE_UNITS_REMAINING,
                "isolated_runtime_confirmed": ISOLATED_RUNTIME_CONFIRMED,
                "planned_total_units": 200,
                "reserve_units": RESERVE_UNITS,
                "scope": "current_runtime_interval_only",
                "resume_requested": RESUME,
            },
            indent=2,
            allow_nan=False,
        )
        + "\n"
    )
else:
    print("Preserving the original starting budget observation for this run folder.")
metadata_path = RUN_DIR / "notebook-run.json"
if metadata_path.exists():
    run_metadata = json.loads(metadata_path.read_text())
    if run_metadata["device"] != DEVICE:
        raise ValueError("Keep the same device for this profiling run folder")
else:
    run_metadata = {
        "notebook": "06_trained_inference",
        "run_id": run_id,
        "device": DEVICE,
        "project_ref": PROJECT_REF,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
# Pin reruns of this run folder to the commit used by the first successful checkout.
EFFECTIVE_PROJECT_REF = run_metadata.get("source_commit", PROJECT_REF)
print("Project revision for this run:", EFFECTIVE_PROJECT_REF)
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError('Set DEVICE to "cpu" or "cuda"')
if COMPUTE_UNITS_REMAINING is not None and COMPUTE_UNITS_REMAINING <= RESERVE_UNITS:
    raise RuntimeError("The 50-unit reserve has been reached. Export the files and disconnect.")
if COMPUTE_UNITS_PER_HOUR is None or COMPUTE_UNITS_REMAINING is None:
    print("Unavailable budget observations are recorded as null.")

Clone or update the project and install dependencies. Logs and exportable storage already exist.
Rerunning this run folder retains its original profiling source revision. The profiler also checks
the unchanged training model code against the saved checkpoint's contract. If setup fails, skip to
the ending-balance and export cells.

In [ ]:
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            'print("x-access-token" if "username" in sys.argv[1].lower() '
            'else os.environ["TRGNN_CLONE_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        clone_environment = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "TRGNN_CLONE_TOKEN": token,
        }
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        run_logged(command, "clone.log", cwd=Path("/content"), env=clone_environment)
except Exception as exc:
    (RUN_DIR / "setup-failure.json").write_text(
        json.dumps({"stage": "clone", "error_type": type(exc).__name__}, indent=2) + "\n"
    )
    raise
finally:
    token = None
    if "clone_environment" in globals():
        clone_environment.pop("TRGNN_CLONE_TOKEN", None)

run_logged(["git", "checkout", EFFECTIVE_PROJECT_REF], "checkout.log")
if EFFECTIVE_PROJECT_REF == "main":
    run_logged(["git", "merge", "--ff-only", "origin/main"], "checkout.log")
run_logged(["git", "rev-parse", "HEAD"], "checkout.log")
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if run_metadata.get("source_commit", source_commit) != source_commit:
    raise RuntimeError("This run folder must retain its original source commit")
run_metadata["source_commit"] = source_commit
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
run_logged(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    "install.log",
)

Check the compiled sampler, GPU and correctness suite before loading data. These commands run in
fresh Python processes after installation. A failed check stops this run. Runtime package versions
must match the saved training contract; validation replay checks the reconstructed model and data.

In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

The fixed workload is in `configs/trained_inference.json`: **all 8,192 parent validation queries**,
primary batch size **32**, secondary sizes **128 and 512**, seed **42**, **5 warmups** and **20
measured repeats**. The same saved weights, occurrence-preserving sampling and feature policy apply
to reference, HGAM and blocks. Accuracy is checked separately from timings.

A graph loaded from the hosted snapshot uses record-time filtering. Historical deletions, merges
and relationship changes remain unresolved. These measurements retain that conditional scope.

In [ ]:
import yaml

INFERENCE_CONFIG = json.loads((PROJECT_ROOT / "configs/trained_inference.json").read_text())
frozen_config_path = RUN_DIR / "frozen-inference-config.json"
if frozen_config_path.exists():
    if json.loads(frozen_config_path.read_text()) != INFERENCE_CONFIG:
        raise RuntimeError("This run folder must retain its original frozen configuration")
else:
    frozen_config_path.write_text(json.dumps(INFERENCE_CONFIG, indent=2) + "\n")
print(json.dumps(INFERENCE_CONFIG, indent=2))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if sha256_file(TRAINING_REPORT) != INFERENCE_CONFIG["training_report_sha256"]:
    raise RuntimeError("The parent training report must match the frozen report SHA256")
parent_report = json.loads(TRAINING_REPORT.read_text())
if parent_report.get("status") != "passed" or not parent_report.get("trained"):
    raise RuntimeError("The parent must be the successful trained notebook 05 run")
if parent_report["checkpoint"]["sha256"] != INFERENCE_CONFIG["checkpoint_sha256"]:
    raise RuntimeError("The parent checkpoint hash differs from the frozen configuration")

pins = json.loads((PROJECT_ROOT / "configs/upstream.json").read_text())
source_receipt = DATA_DIR / ".trgnn-source.json"
try:
    cached_source = json.loads(source_receipt.read_text())
except (OSError, ValueError):
    cached_source = None
required_inputs = [
    DATA_DIR / "manifest.yaml",
    DATA_DIR / "tasks/user-badge/manifest.yaml",
    DATA_DIR / "tasks/user-badge/train.parquet",
    DATA_DIR / "tasks/user-badge/val.parquet",
]
if (DATA_DIR / "manifest.yaml").is_file():
    database_spec = yaml.safe_load((DATA_DIR / "manifest.yaml").read_text())
    required_inputs.extend(DATA_DIR / "db" / f"{name}.parquet" for name in database_spec["tables"])
NEEDS_DOWNLOAD = cached_source != pins["dataset"] or any(not p.is_file() for p in required_inputs)
print("Pinned training/validation inputs need download:", NEEDS_DOWNLOAD)

### Upload the saved notebook 05 checkpoint

Choose the single **`training.pt`** extracted from notebook 05's ZIP. This notebook verifies its
SHA256 before downloading data or launching the profiler. The smaller notebook 04 pilot checkpoint
has a different hash and cannot be used here. There is no training or optimizer update in this run.

On the same VM, the existing verified copy is reused; it is checked again on every execution of
this cell. You can instead set `CHECKPOINT_INPUT` to the path of your own restored file. A profiling
resume needs the completed `batches/` artifacts as well as the input checkpoint. On a fresh VM,
extract your earlier profiling ZIP into temporary storage, set `RESUME = True`, set
`RESUME_ARTIFACTS_DIR` to its `batches/` directory, and set `PROJECT_REF` to its recorded profiling
source commit. Optionally retain its run ID with `RESUME_RUN_ID`. Only flat `batch-*.json` files
are copied into this run; the profiler verifies their contracts and checksums before reuse. Keep
the full earlier ZIP: each fresh VM records a new budget interval and retains original session
information in reused batch artifacts. Missing measurements must be rerun. Source model code,
inputs, runtime versions and hardware are validated again by the profiler.

In [ ]:
checkpoint_candidate = CHECKPOINT_INPUT
if checkpoint_candidate is None and CHECKPOINT_PATH.is_file():
    checkpoint_candidate = CHECKPOINT_PATH
if checkpoint_candidate is None:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose the single training.pt checkpoint from notebook 05's ZIP")
    _, checkpoint_bytes = next(iter(uploaded.items()))
    restored = Path("/content/trgnn-inference-import") / run_id / "training.pt"
    restored.parent.mkdir(parents=True, exist_ok=True)
    restored.write_bytes(checkpoint_bytes)
    checkpoint_candidate = restored
    del checkpoint_bytes, uploaded
checkpoint_candidate = Path(checkpoint_candidate)
if not checkpoint_candidate.is_file():
    raise FileNotFoundError(f"No notebook 05 checkpoint at {checkpoint_candidate}")
checkpoint_sha256 = sha256_file(checkpoint_candidate)
if checkpoint_sha256 != INFERENCE_CONFIG["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint SHA256 does not match the reviewed notebook 05 training.pt")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
if checkpoint_candidate.resolve() != CHECKPOINT_PATH.resolve():
    shutil.copy2(checkpoint_candidate, CHECKPOINT_PATH)
checkpoint_provenance = {
    "sha256": checkpoint_sha256,
    "bytes": CHECKPOINT_PATH.stat().st_size,
    "training_report_sha256": INFERENCE_CONFIG["training_report_sha256"],
    "parent_training_run": INFERENCE_CONFIG["parent_training_run"],
    "use": "fixed_model_inference_only",
}
(RUN_DIR / "checkpoint-input.json").write_text(json.dumps(checkpoint_provenance, indent=2) + "\n")
if RESUME_ARTIFACTS_DIR is not None:
    if not RESUME:
        raise ValueError("RESUME_ARTIFACTS_DIR requires RESUME=True")
    restored_batches = Path(RESUME_ARTIFACTS_DIR)
    if not restored_batches.is_dir():
        raise FileNotFoundError(f"No extracted profiling batches directory at {restored_batches}")
    batch_files = sorted(path for path in restored_batches.glob("batch-*.json") if path.is_file())
    if not batch_files:
        raise ValueError("The extracted profiling directory contains no batch-*.json artifacts")
    destination = RUN_DIR / "batches"
    destination.mkdir(exist_ok=True)
    for artifact in batch_files:
        target = destination / artifact.name
        if target.exists() and target.read_bytes() != artifact.read_bytes():
            raise RuntimeError(f"Conflicting restored batch artifact: {artifact.name}")
        if artifact.resolve() != target.resolve():
            shutil.copy2(artifact, target)
    print("Restored completed-batch candidates:", len(batch_files))
    print("The profiler will verify each candidate before reuse.")
print("Verified notebook 05 checkpoint:", CHECKPOINT_PATH)
print("SHA256:", checkpoint_sha256)

Run the fixed-checkpoint profiler. It reconstructs the parent model, validates the input contract
and replays the parent validation predictions before recording measurements. Paired execution uses
the same sampled occurrences and fixed weights across all three schedules. Correctness checks and
instrumented stage replays stay outside the uninstrumented online totals.

Each completed batch writes an atomic artifact containing raw repeat timings, fingerprints,
correctness checks and memory. The summary records progress. On resume, the profiler reuses only
verified completed artifacts, then measures remaining batches. Preserve all artifacts if a run
fails. No checkpoint is trained or selected using these measurements.

In [ ]:
arguments = [
    sys.executable,
    "-m",
    "trgnn.trained_profile",
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--training-report",
    str(TRAINING_REPORT),
    "--data-dir",
    str(DATA_DIR),
    "--device",
    DEVICE,
    "--batch-sizes",
    *map(str, INFERENCE_CONFIG["batch_sizes"]),
    "--repeats",
    str(INFERENCE_CONFIG["repeats"]),
    "--warmup",
    str(INFERENCE_CONFIG["warmup"]),
    "--seed",
    str(INFERENCE_CONFIG["seed"]),
    "--bootstrap-samples",
    str(INFERENCE_CONFIG["bootstrap_samples"]),
    "--output",
    str(RUN_DIR / "trained-profile.json"),
    "--artifacts-dir",
    str(RUN_DIR / "batches"),
]
if NEEDS_DOWNLOAD:
    arguments.append("--download")
if RESUME:
    arguments.append("--resume")
run_logged(arguments, "trained-profile.log")

Inspect status, reconstruction and parent-validation replay before interpreting timings. Compare
latency, queries per second and memory by batch size. The primary comparison is blocks versus
HGAM at batch size 32; all secondary sizes and failures remain in the report. Use the raw paired
measurements to assess variation. A partial run is evidence of completed batches only.

In [ ]:
report_path = RUN_DIR / "trained-profile.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Status:", report.get("status"))
    print(
        "Completed query batches:",
        report.get("completed_batches", 0),
        "/",
        report.get("expected_batches", "?"),
    )
    replay = report.get("validation_replay", {})
    print("Saved prediction replay:", replay.get("status", "not completed"))
    print("Validation queries checked:", replay.get("validated_query_count", 0))
    print("Replay maximum prediction error:", replay.get("max_absolute_logit_error"))
    for size, summary in report.get("summary_by_batch_size", {}).items():
        print(f"\nBatch size {size}: {summary['query_batches']} query batches")
        for mode, values in summary["modes"].items():
            latency = values["batch_latency_ms"]
            throughput = values["throughput_queries_per_second"]
            print(
                f"  {mode}: median {latency['median']:.3f} ms, "
                f"p95 {latency['p95']:.3f} ms, {throughput:.1f} queries/s"
            )
        comparison = summary["comparisons"]["hgam"]
        reduction = comparison["blocks_latency_reduction_percent"]
        print(f"  Blocks latency reduction versus HGAM: {reduction:+.2f}% (positive is faster)")
        print("  95% interval:", comparison["cluster_bootstrap_95_percent_ci"])
        print("  Maximum prediction errors:", summary.get("max_absolute_logit_error", {}))
    print("GPU activity recording:", report.get("telemetry", {}).get("status", "not started"))
    if "error" in report:
        print("Error:", report["error"])
    print("Keep the complete ZIP: raw measurements and audits are saved alongside this summary.")
else:
    print("No profiler report yet; export the available setup logs and input checkpoint.")

### Record the ending balance

After profiling stops, copy the actual remaining units from Colab's resource panel below. Leave
`None` if unavailable. The report records the account balance change between the two observations,
rounded to Colab's two displayed decimal places. It attributes that change to this runtime interval
only if other runtime usage was checked at the start and no balance increase occurred.

The starting observation stays intact when resuming in this session. A fresh-VM resume starts a
new interval, so retain all earlier archives to review total consumption. Previously reported
balances are prior observations, not a preset for this run. Budget observations are independent of
whether profiling passed; a missing observation does not require repeating the run.

In [ ]:
ENDING_UNITS_REMAINING = None
observed_nonnegative_number("ENDING_UNITS_REMAINING", ENDING_UNITS_REMAINING)
starting_budget = json.loads((RUN_DIR / "colab-budget.json").read_text())
starting_balance = starting_budget["units_remaining_observed"]
account_delta = (
    round(starting_balance - ENDING_UNITS_REMAINING, 2)
    if starting_balance is not None and ENDING_UNITS_REMAINING is not None
    else None
)
isolated_delta = (
    account_delta
    if starting_budget.get("isolated_runtime_confirmed")
    and account_delta is not None
    and account_delta >= 0
    else None
)
ending_budget = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "starting_observation_at_utc": starting_budget.get("recorded_at_utc"),
    "starting_units_remaining_observed": starting_balance,
    "ending_units_remaining_observed": ENDING_UNITS_REMAINING,
    "account_balance_decrease_units": account_delta,
    "balance_delta_precision_decimal_places": 2,
    "isolated_run_consumption_units": isolated_delta,
    "isolated_runtime_confirmed": starting_budget.get("isolated_runtime_confirmed", False),
    "scope": "current_runtime_interval_only",
    "note": (
        "Attribution needs observed balances and an isolated runtime; "
        "earlier intervals are separate."
    ),
}
(RUN_DIR / "budget-end.json").write_text(
    json.dumps(ending_budget, indent=2, allow_nan=False) + "\n"
)
print(json.dumps(ending_budget, indent=2))
if ENDING_UNITS_REMAINING is not None and ENDING_UNITS_REMAINING <= RESERVE_UNITS:
    print("The 50-unit reserve has been reached. Export and disconnect; do not start another run.")

## Download the report, raw measurements and input checkpoint

Run this after success or failure, before disconnecting. The ZIP recursively includes the input
checkpoint, all `batches/` artifacts, frozen configuration, report, metadata and logs from this run
folder. It excludes the database. Confirm the ZIP contains `checkpoints/training.pt` and the
completed batch artifacts. Keep the ZIP for review and any profiling resume. Then **disconnect and
delete the runtime**. Do not leave an idle GPU connected.

In [ ]:
from google.colab import files

archive = shutil.make_archive(
    "/content/trgnn-trained-inference-" + run_id, "zip", root_dir=str(RUN_DIR)
)
files.download(archive)